# Sports Arena: IPL Performance Analyzer

## 1. Introduction

**Objective.** Use a real ball-by-ball dataset to answer three questions about the
Indian Premier League (IPL):
1. **Player form**: how does a player's batting or bowling change across seasons and within a season?
2. **Team comparisons**: which teams win most, score fastest in each phase, and win at home?
3. **Top performers**: who won the Orange Cap and Purple Cap, and who leads each metric?
4. **Predictions**: who is likely to win the 2020 title and the main awards?

**IPL background.** The IPL is a Twenty20 (T20) cricket league started in 2008. Each
innings has at most 20 overs of 6 legal balls. Teams play a league stage and then playoffs.

**Dataset.** *Indian Premier League 2008-2019* by Navaneesh Kumar on Kaggle (https://www.kaggle.com/datasets/nowke9/ipldata, license CC BY-NC-SA 4.0): `matches.csv` (756 matches,
one row per match) and `deliveries.csv` (179,078 rows, one row per ball). Seasons covered: **2008-2019**.
See `data/README.md` for the source, license and checksums.

**Tools.** Only open-source Python libraries: pandas (with NumPy), matplotlib and seaborn.

## 2. Setup and imports

The first cell installs the exact library versions from `requirements.txt`, so everyone gets
the same results. (If Colab shows a *Restart session* button after this cell, click it and
choose **Runtime > Run all** again. This only happens when Colab already had other versions loaded.)

In [ ]:
!pip install -q pandas==2.2.3 matplotlib==3.9.2 seaborn==0.13.2

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# >>>>>>> EDIT THIS ONE LINE: your GitHub username <<<<<<<
GITHUB_USER = "Mohanp1821"
GITHUB_REPO = "sports-arena"

# Charts: show them on screen AND save them as PNG files.
SHOW_CHARTS = True
if os.path.exists("../outputs"):     # running inside the project's notebooks/ folder
    OUTPUT_FOLDER = "../outputs"
else:                                 # running in Colab
    OUTPUT_FOLDER = "outputs"

# Settings you can change
CHOSEN_SEASON = 2016
FORM_BATTER = "V Kohli"
COMPARE_PLAYERS = ['V Kohli', 'RG Sharma']
CAREER_BOWLER = "SL Malinga"
HEAD_TO_HEAD_TEAMS = ['Mumbai Indians', 'Chennai Super Kings']

# Colours: a colour-blind-safe set.
BLUE = "#2a78d6"
ORANGE = "#eb6834"
AQUA = "#1baf7a"
GREY = "#8a8984"
PHASE_COLOURS = {"Powerplay": BLUE, "Middle": ORANGE, "Death": AQUA}

## 3. Load and inspect the data

We load the **cleaned** files made by `src/prepare_data.py`. The code first tries
GitHub (for Colab) and, if that fails, the local `data/processed/` folder.

In [ ]:
def load_csv(file_name):
    """Load one cleaned CSV: first from GitHub, then from local folders."""
    github_url = ("https://raw.githubusercontent.com/" + GITHUB_USER + "/" + GITHUB_REPO
                  + "/main/data/processed/" + file_name)
    local_paths = ["../data/processed/" + file_name, "data/processed/" + file_name]

    # Try GitHub only when the username has been filled in.
    if GITHUB_USER != "<USERNAME>":
        try:
            print("Loading", file_name, "from GitHub ...")
            return pd.read_csv(github_url, parse_dates=["date"], low_memory=False)
        except Exception as error:
            print("  GitHub failed:", error)

    # Fallback: local files.
    for path in local_paths:
        if os.path.exists(path):
            print("Loading", file_name, "from", path)
            return pd.read_csv(path, parse_dates=["date"], low_memory=False)

    raise FileNotFoundError("Could not find " + file_name + ". Set GITHUB_USER in the setup cell.")


matches = load_csv("matches_clean.csv")
deliveries = load_csv("deliveries_clean.csv")

In [ ]:
print("matches   :", matches.shape)
print("deliveries:", deliveries.shape)
print("Seasons   :", sorted(matches["season"].unique()))
matches.head()

In [ ]:
deliveries.head()

In [ ]:
# Data types and missing values in the ball-by-ball file.
# player_dismissed / dismissal_kind are empty on most balls because most balls are not wickets.
info = pd.DataFrame({"dtype": deliveries.dtypes.astype(str),
                     "missing": deliveries.isna().sum()})
info

## 4. Data cleaning recap (what `prepare_data.py` fixed and why)

| # | Problem in the raw data | What we did | Why it matters |
|---|---|---|---|
| 1 | Team renames (Delhi Daredevils, Kings XI Punjab, Royal Challengers Bangalore, "Rising Pune Supergiants") | Mapped to current names with a dictionary; defunct teams kept | One team should be one row in every chart |
| 2 | Same stadium, different spellings (e.g. "M. Chinnaswamy Stadium") | Mapped to one standard name | Home/venue analysis would split one ground into two |
| 3 | Two date formats (`2017-04-05` and `07/04/18`) | Read both formats into one date column | Match ids are not in time order, so form charts must sort by date |
| 4 | Season format | Already plain years (2008-2019); set season = year of match date | Some datasets use "2007/08"; our rule handles both |
| 5 | 4 no-result matches, 19 rain (D/L) matches | Flagged with `no_result` and `rain_affected` columns | Removed from win % and average-score charts |
| 6 | **2018-2019 extras counted twice**: 1,245 wides/byes/leg-byes also stored as batter runs | Batter runs set to 0 on those balls | Warner 2019 was 727 runs, is now the official 692 |
| - | Super overs (81 balls) | Kept in the file, removed in every stat | Official records exclude super overs |

In [ ]:
print("Teams after cleaning:", sorted(set(matches["team1"]) | set(matches["team2"])))
print("No-result matches   :", matches["no_result"].sum())
print("Rain (D/L) matches  :", matches["rain_affected"].sum())
print("Super-over balls    :", (deliveries["is_super_over"] == 1).sum())
print("Overs are numbered  :", deliveries["over"].min(), "to", deliveries["over"].max())

## 5. Derived metrics

All cricket formulas are below. The key rules:
* **Ball faced** (batter) = every ball except a *wide*.
* **Legal ball** (bowler) = not a wide and not a no-ball.
* **Runs conceded** = bat runs + wides + no-balls (byes and leg-byes are not the bowler's fault).
* **Bowler wicket** = bowled, caught, caught and bowled, lbw, stumped, hit wicket (not run out, retired hurt, obstructing the field).
* **Phases**: Powerplay = overs 1-6, Middle = 7-15, Death = 16-20.

In [ ]:
BOWLER_WICKET_KINDS = ['bowled', 'caught', 'caught and bowled', 'lbw', 'stumped', 'hit wicket']


def remove_super_overs(deliveries):
    """
    Remove super over balls. A super over is a one-over tie-breaker.
    Official IPL player records do NOT include super overs, so we leave
    them out of every player and team statistic.
    """
    return deliveries[deliveries["is_super_over"] == 0].copy()


def phase_of_over(over_number):
    """
    Return the match phase for an over number (overs are numbered 1-20):
        Powerplay = overs 1-6   (only 2 fielders allowed outside the circle)
        Middle    = overs 7-15
        Death     = overs 16-20 (the last 5 overs, when batters attack)
    """
    if over_number <= 6:
        return "Powerplay"
    elif over_number <= 15:
        return "Middle"
    else:
        return "Death"


def add_ball_columns(deliveries):
    """
    Add simple True/False and number columns that the stats need.
    We remove super overs first.
    """
    df = remove_super_overs(deliveries)

    # BALL FACED by the batter: every ball EXCEPT a wide.
    # (A wide is too far from the batter to hit, so it is not counted as
    # faced. A no-ball IS counted as faced because the batter can hit it.)
    df["is_ball_faced"] = df["wide_runs"] == 0

    # LEGAL BALL for the bowler: not a wide AND not a no-ball.
    # Wides and no-balls must be bowled again, so they do not count
    # towards the 6 balls of an over.
    df["is_legal_ball"] = (df["wide_runs"] == 0) & (df["noball_runs"] == 0)

    # RUNS CONCEDED by the bowler = runs off the bat + wides + no-balls.
    # Byes and leg-byes are NOT the bowler's fault (the ball missed the bat),
    # so they are not added to the bowler's figures.
    df["runs_conceded"] = df["batsman_runs"] + df["wide_runs"] + df["noball_runs"]

    # BOWLER WICKET: True if the dismissal type is in our list above.
    df["is_bowler_wicket"] = df["dismissal_kind"].isin(BOWLER_WICKET_KINDS)

    # DOT BALL for the bowler: a legal ball where the bowler gave away 0 runs.
    df["is_dot_ball"] = df["is_legal_ball"] & (df["runs_conceded"] == 0)

    # BOUNDARIES hit by the batter.
    df["is_four"] = df["batsman_runs"] == 4
    df["is_six"] = df["batsman_runs"] == 6

    # Phase of the match (Powerplay / Middle / Death).
    df["phase"] = df["over"].apply(phase_of_over)
    return df


def find_player(deliveries, name_part):
    """
    List every player name that contains the text you type.
    The dataset uses short names like "V Kohli" or "MS Dhoni", so use this
    before choosing a player.   Example: find_player(deliveries, "kohli")
    """
    batters = deliveries["batter"].unique().tolist()
    bowlers = deliveries["bowler"].unique().tolist()
    all_names = sorted(set(batters + bowlers))

    matches = []
    for name in all_names:
        if name_part.lower() in name.lower():
            matches.append(name)
    return matches

In [ ]:
# Dataset names are short forms, so search before choosing a player.
print(find_player(deliveries, "kohli"))
print(find_player(deliveries, "dhoni"))

### Batting and bowling metrics

In [ ]:
def batting_innings(deliveries):
    """
    One row per batter per innings: runs, balls faced, match date.
    Used for fifties/hundreds and for the form (rolling average) chart.
    """
    df = add_ball_columns(deliveries)
    innings = df.groupby(["batter", "match_id", "inning", "season", "date"]).agg(
        runs=("batsman_runs", "sum"),
        balls=("is_ball_faced", "sum"),
    ).reset_index()

    # Sort each player's innings in the order they were played.
    innings = innings.sort_values(["batter", "date", "match_id", "inning"])
    return innings.reset_index(drop=True)


def batting_stats(deliveries, group_columns=["batter"]):
    """
    Batting table. By default one row per batter for the whole IPL.
    Use group_columns=["batter", "season"] to get one row per batter per season.

    Formulas (cricket rules):
      runs            = total runs scored off the bat
      balls_faced     = all balls except wides
      innings         = number of innings in which the batter faced a ball
      dismissals      = times the batter was out (retired hurt is NOT out)
      average         = runs / dismissals     (empty if never dismissed)
      strike_rate     = runs / balls_faced * 100  (runs per 100 balls)
      boundary_pct    = runs from 4s and 6s / runs * 100
      fifties         = innings with 50-99 runs;  hundreds = innings with 100+
    """
    df = add_ball_columns(deliveries)

    # Step 1: add up runs, balls, fours, sixes for each group.
    stats = df.groupby(group_columns).agg(
        runs=("batsman_runs", "sum"),
        balls_faced=("is_ball_faced", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
    ).reset_index()

    # Step 2: count innings, fifties and hundreds from the innings table.
    innings = batting_innings(deliveries)
    innings["is_fifty"] = (innings["runs"] >= 50) & (innings["runs"] < 100)
    innings["is_hundred"] = innings["runs"] >= 100
    innings_count = innings.groupby(group_columns).agg(
        innings=("match_id", "count"),
        fifties=("is_fifty", "sum"),
        hundreds=("is_hundred", "sum"),
        highest_score=("runs", "max"),
    ).reset_index()
    stats = stats.merge(innings_count, on=group_columns, how="left")

    # Step 3: count dismissals. The person who is OUT is in "player_dismissed"
    # (this can be the non-striker in a run out, so we cannot use "batter").
    outs = df[df["player_dismissed"].notna()]
    outs = outs[outs["dismissal_kind"] != "retired hurt"]
    outs = outs.rename(columns={"player_dismissed": "out_player"})
    out_group = ["out_player"] + group_columns[1:]   # e.g. ["out_player", "season"]
    dismissals = outs.groupby(out_group).size().reset_index(name="dismissals")
    dismissals = dismissals.rename(columns={"out_player": "batter"})
    stats = stats.merge(dismissals, on=group_columns, how="left")
    stats["dismissals"] = stats["dismissals"].fillna(0).astype(int)

    # Step 4: ratios.
    # Average: if a batter was never out we cannot divide by zero, so we
    # leave the average empty (NaN) instead of showing "infinity".
    stats["average"] = stats["runs"] / stats["dismissals"].replace(0, pd.NA)
    stats["average"] = pd.to_numeric(stats["average"]).round(2)
    stats["strike_rate"] = (stats["runs"] / stats["balls_faced"] * 100).round(2)
    boundary_runs = stats["fours"] * 4 + stats["sixes"] * 6
    stats["boundary_pct"] = (boundary_runs / stats["runs"].replace(0, pd.NA) * 100)
    stats["boundary_pct"] = pd.to_numeric(stats["boundary_pct"]).round(2)
    return stats


def bowling_stats(deliveries, group_columns=["bowler"]):
    """
    Bowling table. By default one row per bowler for the whole IPL.
    Use group_columns=["bowler", "season"] for one row per bowler per season.

    Formulas (cricket rules):
      legal_balls    = balls that are not wides or no-balls
      overs          = legal_balls / 6
      runs_conceded  = bat runs + wides + no-balls (not byes / leg-byes)
      wickets        = bowled, caught, c&b, lbw, stumped, hit wicket
      economy        = runs_conceded / overs         (runs given per over)
      bowling_avg    = runs_conceded / wickets       (runs given per wicket)
      bowling_sr     = legal_balls / wickets         (balls needed per wicket)
      dot_pct        = dot balls / legal_balls * 100
    """
    df = add_ball_columns(deliveries)
    stats = df.groupby(group_columns).agg(
        legal_balls=("is_legal_ball", "sum"),
        runs_conceded=("runs_conceded", "sum"),
        wickets=("is_bowler_wicket", "sum"),
        dot_balls=("is_dot_ball", "sum"),
        matches=("match_id", "nunique"),
    ).reset_index()

    stats["overs"] = (stats["legal_balls"] / 6).round(1)
    stats["economy"] = (stats["runs_conceded"] / (stats["legal_balls"] / 6)).round(2)

    # Bowlers with 0 wickets would cause a divide-by-zero, so leave empty.
    no_zero_wickets = stats["wickets"].replace(0, pd.NA)
    stats["bowling_avg"] = pd.to_numeric(stats["runs_conceded"] / no_zero_wickets).round(2)
    stats["bowling_sr"] = pd.to_numeric(stats["legal_balls"] / no_zero_wickets).round(2)
    stats["dot_pct"] = (stats["dot_balls"] / stats["legal_balls"] * 100).round(2)
    return stats


def death_over_bowling(deliveries, min_overs=20):
    """Bowling stats for overs 16-20 only, for bowlers with at least min_overs there."""
    df = remove_super_overs(deliveries)
    death_balls = df[df["over"] >= 16]
    stats = bowling_stats(death_balls)
    stats = stats[stats["overs"] >= min_overs]
    return stats.sort_values("economy").reset_index(drop=True)

In [ ]:
batting = batting_stats(deliveries)
batting.sort_values("runs", ascending=False).head(10)

In [ ]:
bowling = bowling_stats(deliveries)
bowling.sort_values("wickets", ascending=False).head(10)

### Top performer helpers

In [ ]:
BATTING_METRICS = ['runs', 'strike_rate', 'average', 'sixes', 'fours', 'boundary_pct']
BOWLING_METRICS = ['wickets', 'economy', 'bowling_avg', 'bowling_sr', 'dot_pct']
LOWER_IS_BETTER = ['economy', 'bowling_avg', 'bowling_sr']


def top_performers(deliveries, season, metric, n=10, min_balls=0):
    """
    Return the top n players for one metric.
      season    : a year like 2016, or None for all seasons together
      metric    : e.g. "runs", "strike_rate", "wickets", "economy"
      min_balls : minimum balls faced (batting) or legal balls (bowling),
                  so a player with 2 lucky balls does not top the list
    """
    df = deliveries
    if season is not None:
        df = deliveries[deliveries["season"] == season]

    if metric in BATTING_METRICS:
        table = batting_stats(df)
        table = table[table["balls_faced"] >= min_balls]
        name_column = "batter"
    elif metric in BOWLING_METRICS:
        table = bowling_stats(df)
        table = table[table["legal_balls"] >= min_balls]
        name_column = "bowler"
    else:
        raise ValueError("Unknown metric: " + metric)

    ascending = metric in LOWER_IS_BETTER
    # Sort by the metric; if two players tie, sort by name so the order is fixed.
    table = table.dropna(subset=[metric])
    table = table.sort_values([metric, name_column], ascending=[ascending, True])
    return table.head(n).reset_index(drop=True)


def cap_winners(deliveries):
    """
    Orange Cap = most runs in a season. Purple Cap = most wickets in a season.
    (If two bowlers have the same wickets, the real Purple Cap goes to the
    better economy; we do the same.)
    """
    bat = batting_stats(deliveries, ["batter", "season"])
    bat = bat.sort_values(["season", "runs"], ascending=[True, False])
    orange = bat.groupby("season").head(1)[["season", "batter", "runs"]]

    bowl = bowling_stats(deliveries, ["bowler", "season"])
    bowl = bowl.sort_values(["season", "wickets", "economy"], ascending=[True, False, True])
    purple = bowl.groupby("season").head(1)[["season", "bowler", "wickets", "economy"]]

    caps = orange.merge(purple, on="season")
    caps.columns = ["season", "orange_cap", "runs", "purple_cap", "wickets", "economy"]
    return caps.reset_index(drop=True)


def player_of_match_counts(matches, n=10):
    """Players with the most Player of the Match awards."""
    counts = matches["player_of_match"].value_counts().reset_index()
    counts.columns = ["player", "awards"]
    counts = counts.sort_values(["awards", "player"], ascending=[False, True])
    return counts.head(n).reset_index(drop=True)

### Team metrics

In [ ]:
HOME_VENUES = {
    "Chennai Super Kings": "MA Chidambaram Stadium, Chepauk",
    "Mumbai Indians": "Wankhede Stadium",
    "Kolkata Knight Riders": "Eden Gardens",
    "Royal Challengers Bengaluru": "M Chinnaswamy Stadium",
    "Delhi Capitals": "Feroz Shah Kotla",
    "Punjab Kings": "Punjab Cricket Association IS Bindra Stadium, Mohali",
    "Rajasthan Royals": "Sawai Mansingh Stadium",
    "Sunrisers Hyderabad": "Rajiv Gandhi International Stadium, Uppal"
}


def team_results(matches):
    """
    Turn each match into TWO rows, one for each team, with won = True/False.
    No-result matches are removed because nobody won or lost them.
    """
    played = matches[matches["no_result"] == False]

    # Rows from team1's point of view.
    side1 = played[["match_id", "season", "venue", "winner"]].copy()
    side1["team"] = played["team1"]
    side1["opponent"] = played["team2"]

    # Rows from team2's point of view.
    side2 = played[["match_id", "season", "venue", "winner"]].copy()
    side2["team"] = played["team2"]
    side2["opponent"] = played["team1"]

    results = pd.concat([side1, side2], ignore_index=True)
    results["won"] = results["team"] == results["winner"]
    return results


def team_win_percent(matches, by_season=True):
    """Win % = wins / matches played * 100 (per season, or all-time)."""
    results = team_results(matches)
    group_columns = ["team", "season"] if by_season else ["team"]
    table = results.groupby(group_columns).agg(
        played=("match_id", "count"),
        wins=("won", "sum"),
    ).reset_index()
    table["win_pct"] = (table["wins"] / table["played"] * 100).round(1)
    return table


def team_run_rate(deliveries):
    """
    Team run rate per season = all runs scored (including extras) / overs faced.
    Overs faced = legal balls / 6.
    """
    df = add_ball_columns(deliveries)
    table = df.groupby(["batting_team", "season"]).agg(
        runs=("total_runs", "sum"),
        legal_balls=("is_legal_ball", "sum"),
    ).reset_index()
    table["run_rate"] = (table["runs"] / (table["legal_balls"] / 6)).round(2)
    return table


def phase_run_rate(deliveries):
    """Run rate for each team in each phase (Powerplay / Middle / Death), all seasons."""
    df = add_ball_columns(deliveries)
    table = df.groupby(["batting_team", "phase"]).agg(
        runs=("total_runs", "sum"),
        legal_balls=("is_legal_ball", "sum"),
    ).reset_index()
    table["run_rate"] = (table["runs"] / (table["legal_balls"] / 6)).round(2)
    return table


def first_innings_scores(deliveries, matches):
    """
    Total of the first innings in every match, and the season average.
    Rain-affected (D/L) and no-result matches are removed, because a
    shortened innings would pull the average down unfairly.
    """
    df = remove_super_overs(deliveries)
    first = df[df["inning"] == 1]
    totals = first.groupby(["match_id", "season"])["total_runs"].sum().reset_index()

    good_matches = matches[(matches["no_result"] == False) & (matches["rain_affected"] == False)]
    totals = totals[totals["match_id"].isin(good_matches["match_id"])]

    by_season = totals.groupby("season")["total_runs"].mean().round(1).reset_index()
    by_season = by_season.rename(columns={"total_runs": "avg_first_innings"})
    return by_season


def bat_first_vs_chase(deliveries, matches):
    """
    For each season: % of matches won by the team batting first vs chasing.
    We find the team that batted first from the ball-by-ball data (inning 1),
    which is more reliable than guessing from the toss decision.
    """
    first = deliveries[deliveries["inning"] == 1]
    bat_first = first.groupby("match_id")["batting_team"].first().reset_index()
    bat_first = bat_first.rename(columns={"batting_team": "bat_first_team"})

    played = matches[matches["no_result"] == False]
    played = played.merge(bat_first, on="match_id", how="inner")
    played["bat_first_won"] = played["winner"] == played["bat_first_team"]

    table = played.groupby("season").agg(
        matches=("match_id", "count"),
        bat_first_wins=("bat_first_won", "sum"),
    ).reset_index()
    table["bat_first_win_pct"] = (table["bat_first_wins"] / table["matches"] * 100).round(1)
    table["chase_win_pct"] = (100 - table["bat_first_win_pct"]).round(1)
    return table


def toss_impact(matches):
    """
    Does winning the toss help? Returns the % of matches won by the toss
    winner, overall and split by what they chose (bat or field).
    """
    played = matches[matches["no_result"] == False].copy()
    played["toss_winner_won"] = played["toss_winner"] == played["winner"]
    table = played.groupby("toss_decision").agg(
        matches=("match_id", "count"),
        toss_winner_wins=("toss_winner_won", "sum"),
    ).reset_index()
    table["win_pct"] = (table["toss_winner_wins"] / table["matches"] * 100).round(1)

    overall = played["toss_winner_won"].mean() * 100
    return table, round(overall, 1)


def home_away_performance(matches):
    """Win % at the home ground vs everywhere else, for the 8 major teams."""
    results = team_results(matches)
    results = results[results["team"].isin(HOME_VENUES.keys())].copy()

    # Look up each team's home ground and compare it with the match venue.
    results["home_ground"] = results["team"].map(HOME_VENUES)
    results["location"] = "Away"
    results.loc[results["venue"] == results["home_ground"], "location"] = "Home"

    table = results.groupby(["team", "location"]).agg(
        played=("match_id", "count"),
        wins=("won", "sum"),
    ).reset_index()
    table["win_pct"] = (table["wins"] / table["played"] * 100).round(1)
    return table


def head_to_head(matches, team_a, team_b):
    """
    Head-to-head record: in matches between team_a and team_b only,
    how many did each team win in each season? No-results are left out.
    """
    results = team_results(matches)
    # Keep only rows where team_a played against team_b.
    games = results[(results["team"] == team_a) & (results["opponent"] == team_b)]

    table = games.groupby("season").agg(
        played=("match_id", "count"),
        wins_a=("won", "sum"),
    ).reset_index()
    table["wins_b"] = table["played"] - table["wins_a"]
    table = table.rename(columns={"wins_a": team_a, "wins_b": team_b})
    return table


# The 8 teams that played in almost every season.
MAJOR_TEAMS = list(HOME_VENUES.keys())

In [ ]:
toss_table, toss_overall = toss_impact(matches)
print("The toss winner won", toss_overall, "% of matches")
toss_table

In [ ]:
team_run_rate(deliveries).head(10)

## 6. Player form trends

Chart helpers first: every chart is saved to the `outputs/` folder as a PNG.

In [ ]:
def setup_style():
    """Use one clean, readable style for every chart."""
    sns.set_theme(style="whitegrid", font_scale=1.1)
    plt.rcParams["figure.dpi"] = 100
    plt.rcParams["axes.titleweight"] = "bold"


def save_chart(fig, file_name):
    """Save a chart into outputs/, then show it (notebook) or close it (script)."""
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    path = os.path.join(OUTPUT_FOLDER, file_name)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("  saved", path)
    if SHOW_CHARTS:
        plt.show()
    else:
        plt.close(fig)


def safe_file_name(text):
    """Turn a player name like 'V Kohli' into 'v_kohli' for a file name."""
    return text.lower().replace(" ", "_").replace(".", "")


setup_style()

**Within a season:** bars are runs in each innings; the orange line is the average of the
last 5 innings (a *rolling average*), which smooths out single big or small scores to show form.

In [ ]:
def plot_player_form(deliveries, player, season):
    """Runs in each innings of one season, plus a 5-innings rolling average."""
    innings = batting_innings(deliveries)
    innings = innings[(innings["batter"] == player) & (innings["season"] == season)].copy()
    if len(innings) == 0:
        print("No innings found for", player, "in", season)
        return

    # Number the innings 1, 2, 3 ... in the order they were played.
    innings["innings_number"] = range(1, len(innings) + 1)
    # Rolling average: the mean of the last 5 innings (fewer at the start).
    innings["rolling_avg"] = innings["runs"].rolling(window=5, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(innings["innings_number"], innings["runs"], color=BLUE, alpha=0.6,
           label="Runs in the innings")
    ax.plot(innings["innings_number"], innings["rolling_avg"], color=ORANGE,
            linewidth=2.5, marker="o", label="5-innings rolling average")
    ax.axhline(50, color=GREY, linestyle="--", linewidth=1, label="50 runs")
    ax.set_title(player + ": runs per innings in IPL " + str(season))
    ax.set_xlabel("Innings number in the season")
    ax.set_ylabel("Runs")
    ax.set_xticks(innings["innings_number"])
    ax.legend()
    save_chart(fig, "form_" + safe_file_name(player) + "_" + str(season) + ".png")


plot_player_form(deliveries, FORM_BATTER, CHOSEN_SEASON)

**Across a career:** runs per season (top) and strike rate (bottom). Two panels are used because the two numbers have different scales.

In [ ]:
def plot_player_career(deliveries, player):
    """Season-by-season runs and strike rate (two panels, one measure each)."""
    table = batting_stats(deliveries, ["batter", "season"])
    table = table[table["batter"] == player]

    # Two separate panels, because runs and strike rate have different scales.
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
    ax1.bar(table["season"], table["runs"], color=BLUE)
    ax1.set_title(player + ": IPL career by season")
    ax1.set_ylabel("Runs in season")

    ax2.plot(table["season"], table["strike_rate"], color=ORANGE, marker="o", linewidth=2)
    ax2.set_ylabel("Strike rate")
    ax2.set_xlabel("Season")
    ax2.set_xticks(table["season"])
    save_chart(fig, "career_" + safe_file_name(player) + ".png")


plot_player_career(deliveries, FORM_BATTER)

**A bowler's career:** wickets and economy rate per season.

In [ ]:
def plot_bowler_career(deliveries, bowler):
    """A bowler's economy rate and wickets per season (two panels)."""
    table = bowling_stats(deliveries, ["bowler", "season"])
    table = table[table["bowler"] == bowler]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
    ax1.bar(table["season"], table["wickets"], color=BLUE)
    ax1.set_title(bowler + ": wickets and economy by season")
    ax1.set_ylabel("Wickets")

    ax2.plot(table["season"], table["economy"], color=ORANGE, marker="o", linewidth=2)
    ax2.set_ylabel("Economy (runs per over)")
    ax2.set_xlabel("Season")
    ax2.set_xticks(table["season"])
    save_chart(fig, "bowler_" + safe_file_name(bowler) + ".png")


plot_bowler_career(deliveries, CAREER_BOWLER)

**Compare two players:** change the names in `COMPARE_PLAYERS` (use `find_player` to get exact names).

In [ ]:
def compare_players(deliveries, player_a, player_b):
    """Compare two batters' runs per season on one chart."""
    table = batting_stats(deliveries, ["batter", "season"])

    fig, ax = plt.subplots(figsize=(11, 5))
    colours = [BLUE, ORANGE]
    for i, player in enumerate([player_a, player_b]):
        rows = table[table["batter"] == player]
        ax.plot(rows["season"], rows["runs"], marker="o", linewidth=2,
                color=colours[i], label=player)

    ax.set_title("Runs per season: " + player_a + " vs " + player_b)
    ax.set_xlabel("Season")
    ax.set_ylabel("Runs")
    ax.set_xticks(sorted(table["season"].unique()))
    ax.legend()
    save_chart(fig, "compare_" + safe_file_name(player_a) + "_vs_" + safe_file_name(player_b) + ".png")


compare_players(deliveries, COMPARE_PLAYERS[0], COMPARE_PLAYERS[1])

## 7. Team comparisons

In [ ]:
def plot_season_win_pct(matches, season):
    """Bar chart of each team's win % in one season."""
    table = team_win_percent(matches, by_season=True)
    table = table[table["season"] == season].sort_values("win_pct", ascending=False)

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(table["team"], table["win_pct"], color=BLUE)
    ax.set_title("Team win % in IPL " + str(season))
    ax.set_xlabel("Team")
    ax.set_ylabel("Win %")
    ax.tick_params(axis="x", rotation=40)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")
    save_chart(fig, "team_win_pct_" + str(season) + ".png")


plot_season_win_pct(matches, CHOSEN_SEASON)

In [ ]:
def plot_alltime_win_pct(matches):
    """Horizontal bar chart ranking every team by all-time win %."""
    table = team_win_percent(matches, by_season=False)
    table = table.sort_values("win_pct")

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(table["team"], table["win_pct"], color=BLUE)
    for i in range(len(table)):
        row = table.iloc[i]
        ax.text(row["win_pct"] + 0.5, i, str(row["win_pct"]) + "% (" + str(row["played"]) + " matches)",
                va="center", fontsize=9)
    ax.set_title("All-time win % by team (2008-2019)")
    ax.set_xlabel("Win %")
    ax.set_ylabel("Team")
    ax.set_xlim(0, 75)
    save_chart(fig, "team_win_pct_alltime.png")


plot_alltime_win_pct(matches)

**Phase-wise run rate:** how fast each team scores in the Powerplay, Middle and Death overs.

In [ ]:
def plot_phase_run_rate(deliveries):
    """Grouped bar chart: run rate in Powerplay, Middle and Death overs per team."""
    table = phase_run_rate(deliveries)
    table = table[table["batting_team"].isin(MAJOR_TEAMS)]

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=table, x="batting_team", y="run_rate", hue="phase",
                hue_order=["Powerplay", "Middle", "Death"], palette=PHASE_COLOURS, ax=ax)
    ax.set_title("Run rate by match phase (major teams, 2008-2019)")
    ax.set_xlabel("Team")
    ax.set_ylabel("Run rate (runs per over)")
    ax.tick_params(axis="x", rotation=40)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")
    # Put the legend outside the plot so it does not cover any bars.
    ax.legend(title="Phase", loc="upper left", bbox_to_anchor=(1.01, 1))
    save_chart(fig, "phase_run_rate.png")


plot_phase_run_rate(deliveries)

**Toss analysis:** is it better to bat first or chase? The team batting first is taken from the ball-by-ball data.

In [ ]:
def plot_bat_first_vs_chase(deliveries, matches):
    """Grouped bars: % of matches won batting first vs chasing, per season."""
    table = bat_first_vs_chase(deliveries, matches)

    # Change the table from "wide" to "long" format so seaborn can group it.
    long_table = pd.melt(table, id_vars="season",
                         value_vars=["bat_first_win_pct", "chase_win_pct"],
                         var_name="result", value_name="win_pct")
    long_table["result"] = long_table["result"].replace(
        {"bat_first_win_pct": "Batting first", "chase_win_pct": "Chasing"})

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=long_table, x="season", y="win_pct", hue="result",
                palette={"Batting first": BLUE, "Chasing": ORANGE}, ax=ax)
    ax.axhline(50, color=GREY, linestyle="--", linewidth=1)
    ax.set_title("Batting first vs chasing: win % by season")
    ax.set_xlabel("Season")
    ax.set_ylabel("Win %")
    ax.legend(title="")
    save_chart(fig, "bat_first_vs_chase.png")


plot_bat_first_vs_chase(deliveries, matches)

**Scoring trend:** average first-innings total per season (rain-shortened matches removed).

In [ ]:
def plot_first_innings_trend(deliveries, matches):
    """Line chart of the average first-innings score in each season."""
    table = first_innings_scores(deliveries, matches)

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(table["season"], table["avg_first_innings"], color=BLUE, marker="o", linewidth=2)
    for i in range(len(table)):
        row = table.iloc[i]
        ax.text(row["season"], row["avg_first_innings"] + 1.5, str(row["avg_first_innings"]),
                ha="center", fontsize=9)
    ax.set_title("Average first-innings score by season")
    ax.set_xlabel("Season")
    ax.set_ylabel("Average runs")
    ax.set_xticks(table["season"])
    save_chart(fig, "first_innings_trend.png")


plot_first_innings_trend(deliveries, matches)

**Home advantage:** win % at each team's home ground vs away.

In [ ]:
def plot_home_away(matches):
    """Grouped bars: home vs away win % for the 8 major teams."""
    table = home_away_performance(matches)

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=table, x="team", y="win_pct", hue="location",
                hue_order=["Home", "Away"], palette={"Home": BLUE, "Away": ORANGE}, ax=ax)
    ax.set_title("Home ground vs away win % (2008-2019)")
    ax.set_xlabel("Team")
    ax.set_ylabel("Win %")
    ax.tick_params(axis="x", rotation=40)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")
    ax.legend(title="")
    save_chart(fig, "home_vs_away.png")


plot_home_away(matches)

**Head to head:** wins for each team, season by season, in matches between two rivals. Change `HEAD_TO_HEAD_TEAMS` in the setup cell to compare others.

In [ ]:
def plot_head_to_head(matches, team_a, team_b):
    """Grouped bars: wins for each team, per season, in matches between the two."""
    table = head_to_head(matches, team_a, team_b)
    total_a = table[team_a].sum()
    total_b = table[team_b].sum()

    # Long format: one row per season per team, so seaborn can group the bars.
    long_table = pd.melt(table, id_vars="season", value_vars=[team_a, team_b],
                         var_name="team", value_name="wins")

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=long_table, x="season", y="wins", hue="team",
                palette={team_a: BLUE, team_b: ORANGE}, ax=ax)
    ax.set_title("Head to head: " + team_a + " " + str(total_a) + " - "
                 + str(total_b) + " " + team_b)
    ax.set_xlabel("Season")
    ax.set_ylabel("Wins against the other team")
    ax.set_yticks(range(0, int(long_table["wins"].max()) + 2))
    ax.legend(title="")
    save_chart(fig, "head_to_head.png")


plot_head_to_head(matches, HEAD_TO_HEAD_TEAMS[0], HEAD_TO_HEAD_TEAMS[1])

## 8. Top performers per season

**Orange Cap** = most runs in a season. **Purple Cap** = most wickets (ties broken by economy).

In [ ]:
caps = cap_winners(deliveries)
caps

In [ ]:
def plot_cap_winners(caps):
    """Two panels: Orange Cap runs and Purple Cap wickets, labelled with names."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

    ax1.bar(caps["season"], caps["runs"], color=ORANGE)
    for i in range(len(caps)):
        row = caps.iloc[i]
        ax1.text(row["season"], row["runs"] + 10, row["orange_cap"], ha="center",
                 rotation=90, va="bottom", fontsize=9)
    ax1.set_title("Orange Cap (most runs) and Purple Cap (most wickets) by season")
    ax1.set_ylabel("Runs")
    ax1.set_ylim(0, 1250)

    ax2.bar(caps["season"], caps["wickets"], color="#4a3aa7")
    for i in range(len(caps)):
        row = caps.iloc[i]
        ax2.text(row["season"], row["wickets"] + 0.5, row["purple_cap"], ha="center",
                 rotation=90, va="bottom", fontsize=9)
    ax2.set_ylabel("Wickets")
    ax2.set_xlabel("Season")
    ax2.set_ylim(0, 45)
    ax2.set_xticks(caps["season"])
    save_chart(fig, "cap_winners.png")


plot_cap_winners(caps)

**Top 10 charts** for the chosen season. Minimums stop players with very few balls from topping the list.

In [ ]:
def plot_top_bar(table, name_column, metric, title, x_label, file_name):
    """Generic horizontal bar chart for a top-10 table (best player at the top)."""
    # Reverse the order so the best player is drawn at the top.
    table = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(table[name_column], table[metric], color=BLUE)
    for i in range(len(table)):
        value = table.iloc[i][metric]
        ax.text(value, i, " " + str(value), va="center", fontsize=9)
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Player")
    save_chart(fig, file_name)

In [ ]:
season_text = str(CHOSEN_SEASON)
top_sr = top_performers(deliveries, CHOSEN_SEASON, "strike_rate", n=10, min_balls=200)
plot_top_bar(top_sr, "batter", "strike_rate", "Top 10 strike rates, IPL " + season_text + " (min 200 balls)",
             "Strike rate", "top_strike_rate_" + season_text + ".png")

In [ ]:
top_eco = top_performers(deliveries, CHOSEN_SEASON, "economy", n=10, min_balls=180)   # 30 overs = 180 balls
plot_top_bar(top_eco, "bowler", "economy", "Best economy, IPL " + season_text + " (min 30 overs)",
             "Economy (runs per over, lower is better)", "top_economy_" + season_text + ".png")

In [ ]:
top_six = top_performers(deliveries, CHOSEN_SEASON, "sixes", n=10)
plot_top_bar(top_six, "batter", "sixes", "Most sixes, IPL " + season_text,
             "Sixes", "top_sixes_" + season_text + ".png")

**Quadrant chart:** each dot is a batter (min 1,000 balls). The dashed lines are the medians.
*Elite* = high average and high strike rate; *Anchors* = high average, slower;
*Finishers* = fast but lower average; *Struggling* = below both medians.

In [ ]:
def plot_batting_quadrant(deliveries, min_balls=1000):
    """
    Scatter: batting average (x) vs strike rate (y), with median lines.
    The two median lines split the chart into four groups:
      Elite      = high average AND high strike rate
      Anchors    = high average, lower strike rate (steady, build innings)
      Finishers  = lower average, high strike rate (fast hitters late on)
      Struggling = below the median on both
    """
    table = batting_stats(deliveries)
    table = table[table["balls_faced"] >= min_balls].dropna(subset=["average"])
    median_avg = table["average"].median()
    median_sr = table["strike_rate"].median()

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.scatter(table["average"], table["strike_rate"], s=60, color=BLUE,
               edgecolor="white", linewidth=1)
    ax.axvline(median_avg, color=GREY, linestyle="--")
    ax.axhline(median_sr, color=GREY, linestyle="--")

    # Label only the top 15 run scorers, so names do not overlap too much.
    top = table.sort_values("runs", ascending=False).head(15)
    for i in range(len(top)):
        row = top.iloc[i]
        ax.annotate(row["batter"], (row["average"], row["strike_rate"]),
                    xytext=(5, 4), textcoords="offset points", fontsize=9)

    # Quadrant names in the four corners.
    ax.text(0.98, 0.98, "Elite", transform=ax.transAxes, ha="right", va="top", fontsize=13, weight="bold")
    ax.text(0.98, 0.02, "Anchors", transform=ax.transAxes, ha="right", va="bottom", fontsize=13, weight="bold")
    ax.text(0.02, 0.98, "Finishers", transform=ax.transAxes, ha="left", va="top", fontsize=13, weight="bold")
    ax.text(0.02, 0.02, "Struggling", transform=ax.transAxes, ha="left", va="bottom", fontsize=13, weight="bold")

    ax.set_title("Batting average vs strike rate (min " + str(min_balls) + " balls faced)")
    ax.set_xlabel("Batting average (runs per dismissal)")
    ax.set_ylabel("Strike rate (runs per 100 balls)")
    save_chart(fig, "batting_quadrant.png")


plot_batting_quadrant(deliveries)

**Death-over specialists:** best economy in overs 16-20 (min 20 overs bowled there).

In [ ]:
death = death_over_bowling(deliveries, min_overs=20).head(10)
plot_top_bar(death, "bowler", "economy", "Death-over specialists: best economy in overs 16-20 (min 20 overs)",
             "Economy (lower is better)", "death_over_specialists.png")

**Player of the Match awards** (the column exists in `matches.csv`).

In [ ]:
potm = player_of_match_counts(matches)
plot_top_bar(potm, "player", "awards", "Most Player of the Match awards (2008-2019)",
             "Awards", "player_of_match.png")

## 9. Heatmap: team vs season win %

In [ ]:
def plot_team_season_heatmap(matches):
    """Heatmap: rows = teams, columns = seasons, colour = win %."""
    table = team_win_percent(matches, by_season=True)
    # pivot: turn the long table into a grid of team x season.
    grid = table.pivot(index="team", columns="season", values="win_pct")

    fig, ax = plt.subplots(figsize=(14, 7))
    sns.heatmap(grid, annot=True, fmt=".0f", cmap="Blues", linewidths=1,
                linecolor="white", cbar_kws={"label": "Win %"}, ax=ax)
    ax.grid(False)   # turn off background gridlines so empty cells stay blank
    ax.set_title("Win % by team and season (blank = did not play)")
    ax.set_xlabel("Season")
    ax.set_ylabel("Team")
    save_chart(fig, "heatmap_team_season.png")


plot_team_season_heatmap(matches)

## 10. Predictions for the next season (2020)

The data ends in 2019, so we predict **2020**: the champion and four award winners.
No black box, every step can be worked out on paper:

1. **Form score** = weighted average of the last 3 seasons: last season ×3, the one before ×2, then ×1.
   Example: 600, 400, 300 runs → (3×600 + 2×400 + 1×300) ÷ 6 = **483.3**.
2. **Champion:** each team's strength = its form win %. We **play the whole season 10,000 times**
   (a *Monte Carlo simulation*). In one match team A beats team B with chance A ÷ (A + B); the top 4 go
   to the IPL playoffs (Qualifier 1, Eliminator, Qualifier 2, Final).
3. **Awards:** the active player with the best form score.
4. **Backtest:** predict each season 2011–2019 using only earlier seasons, and compare with what really happened.

In [ ]:
import numpy as np   # installed with pandas; used for random numbers

FORM_WEIGHTS = [3, 2, 1]   # weight of last season, season before, season before that
SIMULATIONS = 10000        # how many times the season is played on the computer
RANDOM_SEED = 42           # fixed seed, so the results are the same every run
NEW_TEAM_STRENGTH = 50.0   # a team with no history gets an average strength (50% wins)
TOP_N = 5                  # how many candidates to show for each award
FIRST_BACKTEST_SEASON = 2011   # first season with 3 earlier seasons of history

# The awards we predict:
#   (award name, which season table, name column, value column)
AWARDS = [
    ("Orange Cap (most runs)", "batting", "batter", "runs"),
    ("Purple Cap (most wickets)", "bowling", "bowler", "wickets"),
    ("Most sixes", "batting", "batter", "sixes"),
    ("Most Player of the Match awards", "potm", "player", "awards"),
]

**Past results and the form score.**

In [ ]:
def season_champions(matches):
    """
    The champion of each season = the winner of the FINAL,
    which is the last match of the season (the latest date).
    """
    finals = matches.sort_values("date").groupby("season").tail(1)
    finals = finals[["season", "winner"]].rename(columns={"winner": "champion"})
    return finals.sort_values("season").reset_index(drop=True)


def season_tables(matches, deliveries):
    """
    One row per player per season, for each award:
      batting : runs and sixes      (from batting_stats)
      bowling : wickets and economy (from bowling_stats)
      potm    : Player of the Match awards won in the season
    """
    batting = batting_stats(deliveries, ["batter", "season"])
    bowling = bowling_stats(deliveries, ["bowler", "season"])
    potm = matches.groupby(["player_of_match", "season"]).size().reset_index(name="awards")
    potm = potm.rename(columns={"player_of_match": "player"})
    return {"batting": batting, "bowling": bowling, "potm": potm}


def weighted_form(table, name_column, value_column, last_season):
    """
    Form score = weighted average of value_column over the 3 seasons up to
    last_season (weights 3, 2, 1). A season the player/team missed is skipped,
    so a young player is not punished for seasons before their debut.
    Only names that played in last_season are kept, so retired players drop out.
    (Limitation: a player who missed last season, like Warner in 2018, drops out too.)
    """
    # Keep only the last 3 seasons.
    first_season = last_season - len(FORM_WEIGHTS) + 1
    recent = table[(table["season"] >= first_season) & (table["season"] <= last_season)].copy()

    # Give each season its weight: last_season -> 3, last_season-1 -> 2, ...
    season_weight = {}
    for years_back in range(len(FORM_WEIGHTS)):
        season_weight[last_season - years_back] = FORM_WEIGHTS[years_back]
    recent["weight"] = recent["season"].map(season_weight)
    recent["weighted_value"] = recent[value_column] * recent["weight"]

    # Weighted average = sum(weight * value) / sum(weights of seasons played)
    form = recent.groupby(name_column).agg(
        weighted_total=("weighted_value", "sum"),
        weight_sum=("weight", "sum"),
    ).reset_index()
    form["form"] = (form["weighted_total"] / form["weight_sum"]).round(1)

    # Keep only names that played in the latest season.
    active = table[table["season"] == last_season][name_column]
    form = form[form[name_column].isin(active)]

    # Best form first; if two are equal, sort by name so the order is fixed.
    form = form.sort_values(["form", name_column], ascending=[False, True])
    return form[[name_column, "form"]].reset_index(drop=True)


def team_strengths(matches, last_season, teams):
    """
    Strength of each team = its form win % (weighted, last 3 seasons).
    teams = the teams that will play in the predicted season.
    """
    win_table = team_win_percent(matches[matches["season"] <= last_season], by_season=True)
    form = weighted_form(win_table, "team", "win_pct", last_season)
    form_by_team = dict(zip(form["team"], form["form"]))

    strengths = {}
    for team in teams:
        # A team that did not play last season (e.g. a new team) gets 50.
        strengths[team] = form_by_team.get(team, NEW_TEAM_STRENGTH)
    return strengths

**Simulating a season.** `play_match` → `simulate_season` → `title_chances` (10,000 seasons).

In [ ]:
def play_match(team_a, team_b, strengths, rng):
    """Play one match. Team A wins with chance  A / (A + B)."""
    chance_a = strengths[team_a] / (strengths[team_a] + strengths[team_b])
    if rng.random() < chance_a:     # rng.random() = a random number between 0 and 1
        return team_a
    return team_b


def simulate_season(strengths, rng):
    """Play one full IPL season. Returns (champion, the 4 playoff teams)."""
    teams = list(strengths.keys())
    wins = {}
    for team in teams:
        wins[team] = 0

    # League stage: every pair of teams plays twice (home and away).
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            for game in range(2):
                winner = play_match(teams[i], teams[j], strengths, rng)
                wins[winner] += 1

    # Points table: sort by wins. A tiny random number breaks ties
    # (in the real IPL, net run rate breaks ties).
    table_score = {}
    for team in teams:
        table_score[team] = wins[team] + rng.random() / 10
    points_table = sorted(teams, key=table_score.get, reverse=True)
    first, second, third, fourth = points_table[0:4]

    # Playoffs (the IPL format used since 2011).
    q1_winner = play_match(first, second, strengths, rng)
    q1_loser = second if q1_winner == first else first
    eliminator_winner = play_match(third, fourth, strengths, rng)
    q2_winner = play_match(q1_loser, eliminator_winner, strengths, rng)
    champion = play_match(q1_winner, q2_winner, strengths, rng)
    return champion, points_table[0:4]


def title_chances(strengths, simulations=SIMULATIONS, seed=RANDOM_SEED):
    """
    Play the season many times and count:
      title_pct   = % of seasons the team won the final
      playoff_pct = % of seasons the team finished in the top 4
    """
    rng = np.random.default_rng(seed)   # random number generator with a fixed seed
    titles = {}
    playoffs = {}
    for team in strengths:
        titles[team] = 0
        playoffs[team] = 0

    for run in range(simulations):
        champion, top4 = simulate_season(strengths, rng)
        titles[champion] += 1
        for team in top4:
            playoffs[team] += 1

    rows = []
    for team in strengths:
        rows.append({
            "team": team,
            "strength": strengths[team],
            "playoff_pct": round(playoffs[team] / simulations * 100, 1),
            "title_pct": round(titles[team] / simulations * 100, 1),
        })
    table = pd.DataFrame(rows)
    table = table.sort_values(["title_pct", "team"], ascending=[False, True])
    return table.reset_index(drop=True)

In [ ]:
def plot_title_chances(chances, season):
    """Horizontal bars: playoff chance and title chance for every team."""
    long_table = pd.melt(chances, id_vars="team", value_vars=["playoff_pct", "title_pct"],
                         var_name="outcome", value_name="chance")
    long_table["outcome"] = long_table["outcome"].replace(
        {"playoff_pct": "Reach playoffs (top 4)", "title_pct": "Win the title"})

    fig, ax = plt.subplots(figsize=(11, 6))
    sns.barplot(data=long_table, y="team", x="chance", hue="outcome",
                palette={"Reach playoffs (top 4)": BLUE, "Win the title": ORANGE},
                ax=ax)
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
    ax.set_title("Predicted IPL " + str(season) + ": chances from " + format(SIMULATIONS, ",")
                 + " simulated seasons")
    ax.set_xlabel("Chance (%)")
    ax.set_ylabel("Team")
    ax.set_xlim(0, 100)
    ax.legend(title="", loc="lower right")
    save_chart(fig, "prediction_title_" + str(season) + ".png")


def plot_award_candidates(candidates, season):
    """One small panel per award: the top candidates and their form scores."""
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    award_names = list(candidates["award"].unique())
    for i in range(len(award_names)):
        ax = axes.flat[i]
        rows = candidates[candidates["award"] == award_names[i]].iloc[::-1]   # best at the top
        colours = [GREY] * (len(rows) - 1) + [ORANGE]        # highlight our pick
        ax.barh(rows["player"], rows["form"], color=colours)
        for j in range(len(rows)):
            ax.text(rows.iloc[j]["form"], j, " " + str(rows.iloc[j]["form"]), va="center", fontsize=9)
        ax.set_title(award_names[i], fontsize=12)
        ax.set_xlabel("Form score (weighted " + rows.iloc[0]["measure"] + " per season)")
        ax.set_xlim(0, rows["form"].max() * 1.2)
    fig.suptitle("Predicted IPL " + str(season) + " award winners (orange = our pick)",
                 fontweight="bold")
    save_chart(fig, "prediction_awards_" + str(season) + ".png")

In [ ]:
tables = season_tables(matches, deliveries)
last_season = int(matches["season"].max())     # 2019
next_season = last_season + 1                   # 2020

teams = sorted(set(matches[matches["season"] == last_season]["team1"]))
strengths = team_strengths(matches, last_season, teams)
chances = title_chances(strengths)
plot_title_chances(chances, next_season)
chances

**Award predictions:** the top 5 candidates for each award (orange bar = our pick).

In [ ]:
def award_candidates(tables, last_season, n=TOP_N):
    """The top n candidates for every award, by form score."""
    rows = []
    for award, table_name, name_column, value_column in AWARDS:
        form = weighted_form(tables[table_name], name_column, value_column, last_season)
        for rank in range(min(n, len(form))):
            rows.append({
                "award": award,
                "rank": rank + 1,
                "player": form.iloc[rank][name_column],
                "form": form.iloc[rank]["form"],
                "measure": value_column,
            })
    return pd.DataFrame(rows)


def actual_award_winners(table, name_column, value_column, season):
    """
    Who really won an award in a season (a list, because players can tie).
    For wickets, a tie goes to the better economy, like the real Purple Cap.
    """
    rows = table[table["season"] == season]
    rows = rows[rows[value_column] == rows[value_column].max()]
    if value_column == "wickets":
        rows = rows[rows["economy"] == rows["economy"].min()]
    return sorted(rows[name_column])


candidates = award_candidates(tables, last_season)
plot_award_candidates(candidates, next_season)
candidates

**Backtest: would this method have worked in the past?** `actual_rank` is where the real winner was in our list
(empty = not in the list, e.g. a player who missed the previous season). A random pick of the champion from
8 teams is right only 12.5% of the time.

In [ ]:
def rank_of(name, ranking):
    """Position (1, 2, 3 ...) of a name in our ranked list, or None if missing."""
    if name in ranking:
        return ranking.index(name) + 1
    return None


def backtest(matches, tables):
    """
    For every past season from 2011, predict it using ONLY the seasons before,
    then compare with the real result. The favourite team is the one with the
    best form win % (the simulation always ranks teams in this same order).
    """
    champions = season_champions(matches)
    rows = []
    last_data_season = int(matches["season"].max())
    for season in range(FIRST_BACKTEST_SEASON, last_data_season + 1):
        history_end = season - 1

        # Champion: rank the teams that actually took part in this season.
        teams = sorted(set(matches[matches["season"] == season]["team1"]))
        strengths = team_strengths(matches, history_end, teams)
        team_ranking = sorted(teams, key=strengths.get, reverse=True)
        actual = champions[champions["season"] == season]["champion"].iloc[0]
        rows.append({
            "season": season, "award": "IPL champion",
            "predicted": team_ranking[0], "actual": actual,
            "actual_rank": rank_of(actual, team_ranking),
        })

        # Awards: rank the players who were active in the season before.
        for award, table_name, name_column, value_column in AWARDS:
            form = weighted_form(tables[table_name], name_column, value_column, history_end)
            player_ranking = list(form[name_column])
            winners = actual_award_winners(tables[table_name], name_column, value_column, season)
            # If players tied for the award, use the best-ranked of them.
            ranks = [rank_of(w, player_ranking) for w in winners if rank_of(w, player_ranking)]
            rows.append({
                "season": season, "award": award,
                "predicted": player_ranking[0], "actual": " / ".join(winners),
                "actual_rank": min(ranks) if ranks else None,
            })

    table = pd.DataFrame(rows)
    # "Int64" = whole numbers that may be empty (a winner who was not in our list).
    table["actual_rank"] = table["actual_rank"].astype("Int64")
    table["correct"] = (table["actual_rank"] == 1).fillna(False)
    table["in_top_5"] = (table["actual_rank"] <= 5).fillna(False)   # empty rank -> False
    return table


def backtest_summary(backtest_table, number_of_teams=8):
    """Hit rate per award, next to a random guess for comparison."""
    summary = backtest_table.groupby("award", sort=False).agg(
        seasons=("season", "count"),
        correct=("correct", "sum"),
        in_top_5=("in_top_5", "sum"),
    ).reset_index()
    summary["correct_pct"] = (summary["correct"] / summary["seasons"] * 100).round(0)
    # Picking a champion at random from 8 teams is right 1 time in 8 (12.5%).
    summary["random_guess_pct"] = None
    summary.loc[summary["award"] == "IPL champion", "random_guess_pct"] = round(100 / number_of_teams, 1)
    return summary


test = backtest(matches, tables)
test

In [ ]:
backtest_summary(test, number_of_teams=len(teams))

## 11. Key insights

1. **Chasing wins more often.** Teams batting second won 55.5% of decided matches (2008-2019), and 61-68% in 2016 and 2019.
2. **Captains learned this.** Toss winners chose to field in between 35% and 68% of matches each season up to 2015, but in 81-83% from 2016 to 2019.
3. **The toss barely matters.** The toss winner won only 52.3% of matches, close to a coin flip.
4. **Scores are rising.** The average first-innings score rose from about 150-166 in 2008-2017 to 172.6 in 2018 and 169.2 in 2019.
5. **The Death overs decide totals.** Teams score about 9.8 runs per over in overs 16-20, compared with about 7.6 in the Powerplay and Middle overs.
6. **Chennai and Mumbai are the most consistent teams**, with the best all-time win % (61.0% and 58.3%).
7. **Home advantage is real**: for example Rajasthan (68% home vs 44% away) and Sunrisers (68% vs 44%). Delhi barely benefit (45% vs 43%).
8. **Gayle is the IPL's biggest hitter** (326 sixes, most Player of the Match awards). Kohli's 973 runs in 2016 is the best single season in the data.
9. **Prediction for 2020:** Chennai Super Kings are favourites (23.7% of 10,000 simulated seasons), just ahead of Mumbai Indians (21.2%). The award picks are DA Warner (Orange Cap), K Rabada (Purple Cap) and AD Russell (most sixes and most Player of the Match awards).

## 12. How to reproduce

**In Google Colab**
1. Open the notebook with the *Open in Colab* badge in the README.
2. Make sure `GITHUB_USER` in section 2 is your GitHub username (it is already set in the repo).
3. Click **Runtime > Run all**. Charts appear below each cell and are saved in `outputs/`.

**On your own computer**
```bash
git clone https://github.com/Mohanp1821/sports-arena.git
cd sports-arena
python -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install -r requirements.txt
python src/verify_data.py          # 1. check the raw data is the original
python src/prepare_data.py         # 2. clean it -> data/processed/
python src/analysis.py             # 3. print tables and save all charts -> outputs/
python tests/test_facts.py         # 4. optional: fact-check against official records
python src/predict.py              # 5. predict the 2020 champion and awards
```